# Goal

Третий шаг на `frostbite_12`, в котором экшн план == коммит. Ветка от `51.1e`.

Перезапуск `56.3e` с новым `target_kl`, который учитывает длину экшн плана (x10 от бенча). Также в этом исследовании починен репортинг `episode_lp` и всех производных от него.

# Control plane

In [30]:
DOCKER_REGISTRY = 'develorium'

if DOCKER_REGISTRY is not None:
    import os
    os.environ['DOCKER_REGISTRY'] = DOCKER_REGISTRY

TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_12.ipynb'
OPTUNA_DECISIVE_METRIC = 'rollout/episode_lp_max_conv'

# @launchit.collect
GRID_SEARCH_SPACE = dict(
    random_seed=list(range(1, 121, 10)),
    parent=[
        '18n_ppo_tr_frostbite_12:195',
    ],
)
# @launchit.stop

if 'GRID_SEARCH_SPACE' in globals():
    import math
    LAUNCHES_COUNT = math.prod(map(len, GRID_SEARCH_SPACE.values()))
    assert LAUNCHES_COUNT > 0
else:
    LAUNCHES_COUNT = 12

# Manual override
# LAUNCHES_COUNT = 4

print(f'{LAUNCHES_COUNT=}')

LAUNCHES_COUNT=12


# set_hyperparameters

In [31]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = False
    HP.general.is_torch_compile = True
    
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    
    wm_parent = '18d_world_model_09:224'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False
    HP.vision_head.is_saving = False

    parent = optuna_trial.suggest_categorical('parent', GRID_SEARCH_SPACE['parent'])
    HP.encoder.parent = dict(model=wm_parent, weights=parent)
    HP.encoder.is_trainable = True
    HP.encoder.is_saving = True
    
    HP.agent.parent = dict(weights=parent, lora=None)
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3, 8, 16, 24, 32]
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, dim_feedforward=1024, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.lora = None
    HP.agent.is_trainable = True
    HP.agent.is_saving = dict(weights=True, lora=False)
    
    # Test params
    HP.test.break_on_level_passed = False
    HP.test.envs_count = 30
    HP.test.env_stories = None # just a standard game
    HP.test.is_torch_amp = True
    
    # Rollout params
    HP.rollout.steps_count = 64 # 64 * 10 (action_plan_length) -> 640 actual steps for each env per rollout
    HP.rollout.env_stories = {
        # Main block
        '0:32':   (None, 'nine_lives'),
    }
    HP.rollout.is_per_step_advantage = True
    HP.rollout.is_torch_amp = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000
    
    HP.ppo.epochs_count = 5 
    HP.ppo.batch_size = 256
    HP.ppo.learn_rate = 'shark_fin(0.00013, 0.00013, 0.075)' # БЕРЕЖНОЕ ОТНОШЕНИЕ К ХОЛОДНОМУ ОПТИМИЗАТОРУ
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.policy_loss = dict(is_per_slot_clipping=False) # stick to behavior on 56.2e
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.034)') 
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.2
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = 0.1 # industry bench 0.015 is for a single action policy, for action plan from 10 actions need to adjust accordingly (nominally x10)
    HP.ppo.norm_adv = True # Toggles advantages normalization
    
    HP.ppo.is_torch_amp = True
    
    # Reporting
    HP.report.rollout_env_groups = None
    HP.report.follow_up_freq = 10

    return HP

# Results

Ничего интересного не вышло. По `levels_passed_mean` - среднячок. По `episode_lp` в отличие от `56.3e` не народилось даже тех, кто до 5-го уровня проходит, мрут на 4-ом. 

Обратил внимание, что `48.3` был запущен на других `learn_rate` и `entropy_loss.coef`. Последнее наиболее актуально, т.к. энтропия здесь чуть выше 1. Т.е. агент выбирает из двух действий. В `48.3` агент выбирал из трех действий. Хочется посмотреть, что будет с чуть более высокими `learn_rate` и `entropy_loss.coef`.

|что|`18n_study_56.3f`|
|---|---|
|episode_lp|<img src="./img/episode_lp_56.3f.png">|
|episode_r|<img src="./img/episode_r_56.3f.png">|
|episode_r|<img src="./img/levels_passed_mean_56.3f.png">|

**Выводы**
1) перезапустить с увеличенным `learn_rate` и `entropy_loss.coef`.

# System

In [32]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [33]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_12.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_12',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_56.3f/18n_study_56.3f.ipynb',
 'optuna_study_name': '18n_study_56.3f',
 'optuna_study_serial': '56.3f',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_56.3f/18n_study_56.3f.optuna'}

In [34]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [35]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [36]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [37]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [38]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while LAUNCHES_COUNT is None or completed_launches_count < LAUNCHES_COUNT:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if LAUNCHES_COUNT is None or (completed_launches_count + len(futures) < LAUNCHES_COUNT):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-11-11 12:16:13,428] A new study created in Journal with name: 18n_study_56.3f


2026.11.11-12:16:13.920500     0.470 >> Model instance registered, version=251
2026.11.11-12:16:13.938517     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch251.ipynb"
2026.11.11-12:16:13.940177     0.002 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12:251"; running launches=1
2026.11.11-12:16:45.261670     0.574 >> Model instance registered, version=252
2026.11.11-12:16:45.275767     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch252.ipynb"
2026.11.11-12:16:45.277926     0.002 >> 5.8 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12:252"; running launches=2
2026.11.11-12:17:16.768551     0.433 >> Model instance registered, version=253
2026.11.11-12:17:16.781747     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch253.ipynb"
2026.11.11-12:17:16.782533     0.005 >> 4.8 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12

In [39]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-11-11 17:21:31,218] Using an existing study with name '18n_study_56.3f' instead of creating a new one.


2026.11.11-17:21:31.220176     5.037 >> Study statistics: 
2026.11.11-17:21:31.222205     0.002 >> 	Number of finished trials: 12
2026.11.11-17:21:31.222917     0.001 >> 	Number of pruned trials: 0
2026.11.11-17:21:31.223728     0.001 >> 	Number of complete trials: 12
2026.11.11-17:21:31.224822     0.001 >> Best trial:
2026.11.11-17:21:31.225841     0.001 >> 	Value: 1099.9302325581396
2026.11.11-17:21:31.226430     0.001 >> 	Model version: 256
2026.11.11-17:21:31.226907     0.000 >> 	Params: 
2026.11.11-17:21:31.227375     0.000 >> 		random_seed: 81
2026.11.11-17:21:31.227731     0.000 >> 		parent: 18n_ppo_tr_frostbite_12:195


In [40]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.11.11-17:21:31.235955     0.008 >> Top trials:
2026.11.11-17:21:31.236730     0.001 >> 1) 18n_ppo_tr_frostbite_12:256, value=1099.9302325581396
2026.11.11-17:21:31.237086     0.000 >> 2) 18n_ppo_tr_frostbite_12:257, value=1098.956224350205
2026.11.11-17:21:31.237375     0.000 >> 3) 18n_ppo_tr_frostbite_12:252, value=1098.9316005471956


In [41]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(1098.4949840401277), np.float64(0.5705973566059083))